# 🌍 Atlas Horizon Group — Proyecto BI · Parte 2

- **Nuestra Empresa:** Atlas Horizon Group — Operador Global de Expediciones de Lujo y Logística Crítica  
- **Dataset seleccionado:** `listings_feb_2020.csv`
- **Realizado por:**
    *    Brahim Nouni Ros
    *    Raúl Fulgencio Belmonte  






# Índice

*   [1.1. Análisis exploratorio del juego de datos](#1.1.-Análisis-exploratorio-del-juego-de-datos)
    *   [1.1.1 Respuestas al Análisis Exploratorio](#1.1.1-Respuestas-al-Análisis-Exploratorio)
    *   [1.1.2 Estadísticas Descriptivas y Visualizaciones Exploratorias](#1.1.2-Estadísticas-Descriptivas-y-Visualizaciones-Exploratorias)
*   [1.2. Conexión de los datos con proceso de negocio](#1.2.-Conexión-de-los-datos-con-proceso-de-negocio)
    *   [1.2.1. Origen y Aplicación Concreta de los Datos en el Proyecto](#1.2.1.-Origen-y-Aplicación-Concreta-de-los-Datos-en-el-Proyecto)
    *   [1.2.2 Relevancia del Dataset para Atlas Horizon Group: Un Caso de Negocio Adaptado](#1.2.2-Relevancia-del-Dataset-para-Atlas-Horizon-Group:-Un-Caso-de-Negocio-Adaptado)
*   [1.3. Limpieza y Pre-procesado de los Datos](#1.3.-Limpieza-y-Pre-procesado-de-los-Datos)
*   [2. Reporte de Negocio: Rendimiento y Ocupación de Alojamientos para Expediciones](#2.-Reporte-de-Negocio:-Rendimiento-y-Ocupación-de-Alojamientos-para-Expediciones)
*   [3. Generación del Dataset Limpio y el Informe HTML](#3.-Generación-del-Dataset-Limpio-y-el-Informe-HTML)

Para comenzar, vamos a importar las librerías y cargar el dataset que hemos seleccionado, luego haremos una inspección inicial sobre los datos.

In [ ]:

#importamos las librerias y cargamos el dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("/content/listings_feb_2020.csv")


# 1.1. Análisis exploratorio del juego de datos


Vamos a responder a las preguntas propuestas en el enunciado:
- ¿Cuántos registros tengo?
- ¿Qué tipos de datos son mis variables y si el tipo de columna es correcto?
- ¿Tengo valores perdidos, anómalos, de calidad baja o que haya que recodificar? ¿Cómo lo
vas a gestionar?
- ¿Me interesan todas las filas y columnas, o voy a tener que descartar algunas de ellas?
- ¿Están los datos en el formato adecuado o voy a tener que re-organizarlos de otra forma?
- ¿Tengo los datos suficientes o voy a tener que integrar datos de alguna fuente adicional?

In [ ]:
# Información general del dataset
print("--- Perfilado Inicial ---")
print(f"Número de registros (expediciones): {df.shape[0]}")
print(f"Número de variables: {df.shape[1]}")
print("\n--- Tipos de datos por columna ---")
print(df.dtypes)

In [ ]:
# Inspección de las primeras filas
df.head()

In [ ]:


# 1. Cantidad de registros y columnas
print(f"Dimensiones del dataset: {df.shape[0]} filas y {df.shape[1]} columnas.")

# 2. Análisis de tipos de datos
print("\n--- Tipos de datos detectados ---")
print(df.dtypes)

# 3. Identificación de valores nulos
print("\n--- Valores nulos por columna ---")
missing = df.isnull().sum()
print(missing[missing > 0])

# 4. Estadísticas básicas para identificar valores anómalos
print("\n--- Resumen estadístico de variables clave ---")
print(df[['price', 'availability_365', 'number_of_reviews']].describe())

##1.1.1 Respuestas al Análisis Exploratorio

Aquí respondemos a las preguntas planteadas en la sección 1.1. basadas en la inspección inicial de los datos:

*   **¿Cuántos registros tengo?**
    *   El dataset cuenta con un total de **87,571 registros** (expediciones/listings) y **16 variables**.

*   **¿Qué tipos de datos son mis variables y si el tipo de columna es correcto?**
    *   La mayoría de las variables numéricas (`id`, `host_id`, `price`, `minimum_nights`, `number_of_reviews`, `calculated_host_listings_count`, `availability_365`) están correctamente identificadas como enteros (`int64`).
    *   Las variables `latitude`, `longitude`, `reviews_per_month` son de tipo flotante (`float64`).
    *   Las columnas `name`, `host_name`, `neighbourhood`, `room_type` son de tipo objeto (`object`), lo cual es adecuado para datos categóricos o de texto.
    *   La columna `last_review` es de tipo `object` (texto) y **deberá convertirse a tipo fecha (`datetime`)** para permitir análisis temporales.
    *   La columna `neighbourhood_group` es de tipo `float64` pero, como veremos a continuación, no contiene valores útiles.

*   **¿Tengo valores perdidos, anómalos, de calidad baja o que haya que recodificar? ¿Cómo lo vas a gestionar?**
    *   **Valores perdidos:**
        *   `neighbourhood_group`: Tiene 87,571 valores nulos, lo que representa el **100% de la columna**. Esta columna no aporta información y **será eliminada**.
        *   `last_review` y `reviews_per_month`: Ambas tienen 20,651 valores nulos. Esto suele ocurrir en listings sin reseñas. Para `reviews_per_month`, se podría **imputar con 0** para indicar que no ha habido reseñas.
        *   `name` (26 nulos) y `host_name` (14 nulos): Son un número muy pequeño de nulos que pueden ser gestionados eliminando las filas correspondientes o imputando con un valor como 'Desconocido', dependiendo de su relevancia para el análisis.
    *   **Anomalías/Valores a recodificar:**
        *   `price`: El valor mínimo es 0, lo cual podría indicar listings gratuitos, errores de datos o datos de prueba que requerirán una revisión o eliminación.
        *   `availability_365`: El valor mínimo es 0, lo que significa que algunas expediciones no tienen disponibilidad. Esto es esperable, pero es importante tenerlo en cuenta.

*   **¿Me interesan todas las filas y columnas, o voy a tener que descartar algunas de ellas?**
    *   Como se mencionó, la columna `neighbourhood_group` **será descartada** debido a que carece de datos. Otras columnas como `name` y `host_name` podrían ser evaluadas para su eliminación o anonimización según los requisitos de privacidad o la estrategia de análisis.
    *   Actualmente, todas las filas parecen relevantes para el contexto de expediciones.

*   **¿Están los datos en el formato adecuado o voy a tener que re-organizarlos de otra forma?**
    *   La columna `last_review` **requiere conversión a formato `datetime`**.
    *   Podría ser necesario **reorganizar o recodificar** la variable `room_type` para ajustarla a las dimensiones de negocio de Atlas Horizon Group (e.g., 'Etnográfica', 'Científica', 'Aventura').

*   **¿Tengo los datos suficientes o voy a tener que integrar datos de alguna fuente adicional?**
    *   Para los KPIs principales como el Ticket promedio y la Ocupación, los datos actuales son suficientes.
    *   Sin embargo, para el indicador NPS (Net Promoter Score) o la **satisfacción**, se podría considerar **integrar o simular una columna** basada en `number_of_reviews` o en una combinación de métricas para cubrir este objetivo, como se mencionó previamente.

##1.1.2 Estadísticas Descriptivas y Visualizaciones Exploratorias

Como se sugiere en el enunciado, vamos a implementar diversas estadísticas descriptivas de las distintas variables y visualizaciones
exploratorias, para que os permitan entender correctamente los datos recibidos

In [ ]:
print('--- Estadísticas Descriptivas Completas (Numéricas y Categóricas) ---')
display(df.describe(include='all'))

### Visualización de la Distribución de Precios

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['price'], bins=50, kde=True)
plt.title('Distribución de Precios')
plt.xlabel('Precio')
plt.ylabel('Frecuencia')
plt.xlim(0, df['price'].quantile(0.99)) # Limitar el eje X para mejor visualización (excluir valores extremos)
plt.show()

### Visualización de Tipos de Habitación

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(y='room_type', data=df, order = df['room_type'].value_counts().index)
plt.title('Distribución de Tipos de Habitación')
plt.xlabel('Conteo')
plt.ylabel('Tipo de Habitación')
plt.show()

## 1.2. Conexión de los datos con proceso de negocio

Para Atlas Horizon Group, este dataset de listings de alojamiento puede estar directamente relacionado con el proceso de negocio de **Gestión de la Oferta de Alojamientos para Expediciones**. Este proceso abarca desde la identificación y captación de alojamientos adecuados, su disponibilidad, precios, hasta la gestión de reservas y la experiencia del cliente.

### Proceso de Negocio Generador de los Datos

El dataset `listings_feb_2020.csv`  proviene de una plataforma Airbnb donde  listan y gestionan los alojamientos que ofrecen a sus clientes para las expediciones. Esto implica:

*   **Creación de Listings:** Cada registro (`id`, `name`, `host_id`, `host_name`, `neighbourhood`, `room_type`, `price`) representa una oferta de alojamiento.
*   **Gestión de la Disponibilidad:** `minimum_nights` y `availability_365` indican las reglas y la disponibilidad de cada alojamiento.
*   **Registro de la Actividad y Calidad:** `number_of_reviews`, `last_review`, `reviews_per_month` son métricas que reflejan la actividad y, potencialmente, la calidad o popularidad de los listings.

### Reporte Orientado al Proceso de Negocio

Un reporte clave podría ser el **"Informe de Rendimiento y Ocupación de Alojamientos"**. Este reporte permitiría a Atlas Horizon Group:

*   Evaluar qué tipos de alojamientos (e.g., `room_type`) son más populares o generan más ingresos.
*   Identificar las áreas (`neighbourhood`) con mayor o menor demanda.
*   Monitorear la disponibilidad de los alojamientos y ajustar estrategias de captación.
*   Analizar el impacto del precio en la ocupación y las reseñas.
*   Detectar alojamientos con bajo rendimiento o aquellos que requieren mejoras.

### Modelado Multidimensional: Hecho, Dimensiones y Medidas

Para el análisis multidimensional de este proceso, podemos definir:

*   **Hecho (Fact):** El **`Alojamiento`** o la **`Oferta de Alojamiento`** en un momento dado.
    *   La granularidad principal del dataset es **por cada listing (ID de alojamiento único)**.

*   **Dimensiones:**
    *   **Dimensión `Tiempo`:** Relacionada con `last_review` (fecha de la última reseña), que podría expandirse a `año`, `mes`, `día` para analizar tendencias temporales.
    *   **Dimensión `Alojamiento`:** `id`, `name`, `room_type` (tipo de habitación, que podría recodificarse a 'Tipo de Expedición' como 'Etnográfica', 'Científica', 'Aventura'), `latitude`, `longitude`.
    *   **Dimensión `Anfitrión`:** `host_id`, `host_name`.
    *   **Dimensión `Ubicación`:** `neighbourhood`.

*   **Medidas (Measures):** Son los valores numéricos que nos interesan analizar y agregar:
    *   `price`: Precio del alojamiento.
    *   `minimum_nights`: Noches mínimas requeridas.
    *   `number_of_reviews`: Número total de reseñas (podría ser un proxy para la popularidad o satisfacción).
    *   `reviews_per_month`: Frecuencia de reseñas.
    *   `calculated_host_listings_count`: Número de alojamientos que tiene el anfitrión.
    *   `availability_365`: Disponibilidad del alojamiento (días disponibles en un año).

### Granularidad

La granularidad actual del dataset es a nivel de **`Listing Individual por Fecha de Consulta`** (Febrero 2020). Para el reporte propuesto, podríamos mantener esta granularidad para detalles específicos o agregarla a niveles superiores (e.g., por `Tipo de Habitación`, `Barrio`, `Anfitrión`) para obtener métricas consolidadas.

##  1.2.1. Origen y Aplicación Concreta de los Datos en el Proyecto

Este dataset, `listings_feb_2020.csv`, representa la oferta de alojamientos disponibles en Londres durante febrero de 2020. Dada la naturaleza de Atlas Horizon Group como operador de expediciones de lujo, podemos inferir que estos datos provienen de:

1.  **Plataformas de Alojamientos Externas:** Atlas Horizon Group podría utilizar plataformas tipo Airbnb (o similares especializadas en alojamientos únicos y de alta gama) para identificar y seleccionar hospedajes que complementen sus expediciones. Este dataset sería una exportación de los listings que cumplen ciertos criterios de calidad o ubicación.
2.  **Sistema Interno de Gestión de Alojamientos:** Podría ser un registro de los alojamientos que Atlas Horizon Group ya tiene pre-seleccionados o con los que colabora para ofrecer a sus clientes, y que son actualizados periódicamente desde fuentes externas.

### Uso del Dataset en Nuestro Proyecto para Atlas Horizon Group

Utilizaremos este dataset como la base para el análisis y la toma de decisiones, centrándonos en la optimización de la oferta de alojamientos para las expediciones. Concretamente, se aplicará para:

*   **Cálculo de KPIs Clave:**
    *   **Ticket Promedio (PI_COM_01):** La variable `price` será fundamental para determinar el valor promedio de las estancias ofrecidas. Esto nos permitirá evaluar la competitividad y la adecuación de los precios a nuestro segmento de lujo.
    *   **Ocupación (PI_LOG_01):** `availability_365` nos dará una idea de cuántos días al año un alojamiento está disponible. Aunque este dataset no ofrece datos directos de ocupación (reservas realizadas), nos permite inferir la *capacidad de oferta* y su potencial para ser ocupada. Una baja disponibilidad podría indicar alta demanda o exclusividad.
    *   **Satisfacción (PI_SAF_02 - NPS):** `number_of_reviews` y `reviews_per_month` serán utilizados como **proxies para la satisfacción del cliente**. Un mayor número y frecuencia de reseñas, junto con la información de `last_review`, pueden correlacionarse con una mayor popularidad y, por extensión, satisfacción. Como se mencionó, se podría considerar simular una columna de NPS basada en estas métricas para una mejor aproximación al KPI.

*   **Optimización de la Cartera de Alojamientos:** Analizaremos las variables como `room_type`, `neighbourhood`, `latitude`, y `longitude` para identificar patrones geográficos, tipos de alojamientos más populares o rentables, y áreas donde Atlas Horizon Group podría expandir o consolidar su oferta.

*   **Identificación de Oportunidades y Riesgos:** Los valores atípicos en `price` o en `availability_365` podrían señalar oportunidades (alojamientos de lujo infravalorados) o riesgos (precios excesivamente altos, o alojamientos con disponibilidad siempre nula indicando problemas). La columna `calculated_host_listings_count` puede indicar la capacidad o el tipo de anfitrión (individual vs. profesional con múltiples propiedades).

En resumen, este dataset nos permitirá construir una visión analítica de la oferta de alojamientos, crucial para informar las estrategias de **Gestión de la Oferta de Alojamientos para Expediciones** de Atlas Horizon Group, garantizando que se alineen con su propuesta de valor de lujo y logística crítica.

## 1.2.2 Relevancia del Dataset para Atlas Horizon Group: Un Caso de Negocio Adaptado

Aunque el dataset `listings_feb_2020.csv` proviene de un contexto urbano de alojamientos tipo Airbnb, su valor para Atlas Horizon Group, una empresa de expediciones de lujo en ubicaciones remotas, radica en la **universalidad de los principios de gestión de la oferta y la demanda** que subyacen en sus datos.

### Relevancia de los Alojamiento y Experiencia del Cliente:

Atlas Horizon Group organiza expediciones de lujo que, incluso en los entornos más remotos, requieren una gestión meticulosa de la logística, lo que incluye la provisión de alojamientos de alta calidad (desde campamentos base de lujo hasta refugios especializados). Este dataset, aunque no directamente de vuestras propiedades, proporciona un modelo estructurado de cómo se articulan y presentan las ofertas de alojamiento en una plataforma, incluyendo variables clave como precios, tipos de 'habitaciones' (que podemos reinterpretar como 'tipos de experiencias' o 'modalidades de alojamiento en expedición'), disponibilidad y la retroalimentación de los usuarios a través de reseñas.

### Métricas Transferibles y Adaptables:

Las variables contenidas en el dataset son, en esencia, análogas a las métricas que Atlas Horizon Group necesita gestionar para su propia oferta de expediciones:

*   **`price` (precio):** Es un indicador fundamental para el **Ticket Promedio (PI_COM_01)**. Permite analizar la estructura de costos y la estrategia de precios de vuestros servicios de lujo, adaptándolos para asegurar la competitividad y la percepción de valor.
*   **`availability_365` (disponibilidad):** Directamente aplicable al KPI de **Ocupación (PI_LOG_01)**. Aunque en el dataset indica la disponibilidad de un listado, para AHG puede modelar la capacidad de las expediciones o la disponibilidad de los alojamientos exclusivos que ofrecéis, permitiendo una planificación logística y de recursos más eficiente.
*   **`number_of_reviews` y `reviews_per_month` (número y frecuencia de reseñas):** Aunque no es un **NPS (PI_SAF_02)** directo, estas variables sirven como **proxies cruciales para la satisfacción y la popularidad** de una oferta. Para AHG, un alto número y frecuencia de reseñas positivas (o la ausencia de ellas) puede indicar la percepción de calidad y exclusividad de vuestras expediciones y alojamientos. La simulación de un NPS basada en estas métricas es una estrategia viable.

### Análisis Estratégico y Toma de Decisiones:

El análisis de este dataset permite:

*   **Optimización de la Cartera de Ofertas:** Al analizar variables como `room_type` (recodificado a 'tipo de expedición' como 'Etnográfica', 'Científica', 'Aventura'), `neighbourhood` (interpretado como 'región de expedición' o 'base de operaciones'), `latitude` y `longitude`, podemos identificar patrones geográficos, los tipos de ofertas más populares o rentables, y áreas donde Atlas Horizon Group podría expandir o consolidar su oferta de expediciones.
*   **Identificación de Oportunidades y Riesgos:** Los valores atípicos en `price` o en `availability_365` pueden señalar oportunidades (expediciones de lujo con potencial de crecimiento o infravaloradas) o riesgos (costos excesivamente altos o problemas de logística que impiden la disponibilidad). La columna `calculated_host_listings_count` puede indicar la capacidad o el tipo de proveedor de servicios (individual vs. una entidad con múltiples recursos).

En conclusión, este dataset, aunque de un origen aparentemente distinto, proporciona una **infraestructura de datos y métricas análoga** que Atlas Horizon Group puede adaptar para optimizar su **Gestión de la Oferta de Alojamientos para Expediciones**. Permite aplicar principios de gestión de la oferta, fijación de precios y análisis de la satisfacción del cliente, fundamentales para mantener y mejorar la propuesta de valor de lujo y logística crítica que caracteriza a la empresa.

## 1.3. Limpieza y Pre-procesado de los Datos

En este apartado nos enfocamos en preparar los datos para su consumo en el reporte y cuadro de mando, realizando las tareas de Extracción, Transformación y Carga (ETL) necesarias.

### 1. Eliminación de Columnas Innecesarias

Como se identificó en el análisis exploratorio, la columna `neighbourhood_group` tiene el 100% de valores nulos, por lo que no aporta información útil y será eliminada del DataFrame.

In [ ]:
# Eliminar la columna 'neighbourhood_group'
df = df.drop(columns=['neighbourhood_group'])
print(f"Columna 'neighbourhood_group' eliminada. Nuevas dimensiones del dataset: {df.shape[0]} filas y {df.shape[1]} columnas.")

### Gestión de Valores Nulos

Abordaremos los valores nulos en `reviews_per_month`, `last_review`, `name` y `host_name`.

In [ ]:
# Imputar valores nulos en 'reviews_per_month' con 0
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

# Convertir 'last_review' a tipo datetime, los errores serán NaT (Not a Time)
df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')

# Eliminar filas donde 'name' o 'host_name' son nulos (pocos registros)
df.dropna(subset=['name', 'host_name'], inplace=True)

print("Después de imputar y eliminar nulos en columnas específicas, los valores nulos restantes son:")
print(df.isnull().sum())

### Revisión de Valores Anómalos: `price` igual a 0

Verificamos cuántos registros tienen un precio de 0, ya que esto podría indicar datos erróneos o listings gratuitos que no son relevantes para el análisis de precios en expediciones de lujo.

In [ ]:
# Contar registros con precio 0
zero_price_listings = df[df['price'] == 0].shape[0]
print(f"Número de listings con precio 0: {zero_price_listings}")

# Si hay muchos, se podría decidir eliminarlos o gestionarlos de otra forma.
# Por ahora, simplemente los identificamos.

### Recodificación de `room_type` (Preparación)

La recodificación de `room_type` para alinearla con nuestras dimensiones de negocio de Atlas Horizon Group (Etnográfica, Científica, Aventura) es un paso más complejo que requiere una definición clara de cómo se mapean los tipos de habitación actuales a estas nuevas categorías. Por ahora, mostraremos los valores únicos actuales para planificar esta recodificación.

In [ ]:
print("Valores únicos actuales de 'room_type':")
print(df['room_type'].value_counts())

### Recodificación de `room_type` a `expedition_type`

Basándonos en la naturaleza de Atlas Horizon Group (expediciones de lujo) y la necesidad de categorizar los alojamientos según tipos de experiencia ('Etnográfica', 'Científica', 'Aventura'), realizaremos la siguiente recodificación. Es importante destacar que esta es una interpretación inicial y podría ajustarse a las definiciones exactas de vuestras expediciones.

*   **'Entire home/apt'** se recodificará como **'Aventura'**: Asumiendo que estos alojamientos ofrecen una experiencia más independiente o como base exclusiva para una expedición.
*   **'Private room'** se recodificará como **'Científica'**: Pensando en alojamientos individuales que podrían servir como base para investigación o pernoctación de participantes.
*   **'Shared room'** se recodificará como **'Etnográfica'**: Sugiriendo una experiencia más comunitaria o de inmersión cultural, a menudo asociada a alojamientos compartidos.

Crearemos una nueva columna, `expedition_type`, para almacenar esta categorización.

In [ ]:
# Definir el mapeo de room_type a expedition_type
room_type_mapping = {
    'Entire home/apt': 'Aventura',
    'Private room': 'Científica',
    'Shared room': 'Etnográfica'
}

# Aplicar el mapeo a una nueva columna 'expedition_type'
df['expedition_type'] = df['room_type'].map(room_type_mapping)

print("Valores únicos y su conteo para la nueva columna 'expedition_type':")
print(df['expedition_type'].value_counts())

print("\nPrimeras filas con la nueva columna 'expedition_type':")
display(df[['room_type', 'expedition_type']].head())

### Verificación del dataset después del proceso ETL

In [ ]:
print("--- Información del DataFrame después de la limpieza ---")
df.info()
print("\n--- Primeras 5 filas del DataFrame limpio ---")
display(df.head())

## 2. Reporte de Negocio: Rendimiento y Ocupación de Alojamientos para Expediciones

En este apartado, desarrollaremos un reporte orientado al proceso de negocio de **Gestión de la Oferta de Alojamientos para Expediciones** de Atlas Horizon Group, identificando los puntos clave solicitados.

### 2.1. Usuario Final

El reporte está diseñado para:

*   **Responsables de Operaciones y Logística:** Encargados de la planificación y ejecución de las expediciones.
*   **Equipo de Gestión de Producto/Oferta:** Quienes definen y optimizan la cartera de alojamientos disponibles.
*   **Gerencia General:** Para una visión estratégica del rendimiento de la oferta de alojamiento.

El lenguaje y las visualizaciones serán claros, concisos y orientados a resultados, minimizando la jerga técnica y enfocándose en métricas clave de negocio.

### 2.2. Propósito

Este reporte ayudará a:

*   **Evaluar la eficiencia de la oferta de alojamientos:** Identificando qué tipos de 'expedición' (Aventura, Científica, Etnográfica) y ubicaciones (neighbourhood) tienen mejor rendimiento en términos de precio y disponibilidad.
*   **Optimizar la estrategia de precios:** Analizando la distribución de precios y su impacto en la disponibilidad.
*   **Detectar oportunidades y riesgos:** Como alojamientos infravalorados o con baja disponibilidad que requieran atención.
*   **Informar decisiones sobre expansión o ajuste de la cartera:** Basado en la popularidad y el rendimiento de los diferentes tipos de alojamiento.

### 2.3. Frecuencia

El reporte será de tipo **Push (periódico)**, generado **mensualmente**. Esto permitirá a los equipos mantenerse actualizados sobre las tendencias y el rendimiento de la oferta de alojamientos, reaccionando de manera proactiva a cambios en el mercado o en la demanda. Adicionalmente, se podría considerar una versión **pull (bajo demanda)** para análisis más específicos o ad-hoc.

### 2.4. Adopción

Para garantizar la adopción, el reporte se distribuirá vía correo electrónico en formato HTML, que es fácilmente accesible y visualizable sin necesidad de software especializado. Se realizarán sesiones de capacitación con los equipos de Operaciones, Logística y Gestión de Producto para explicar el contenido, las métricas y cómo interpretar la información para la toma de decisiones. Se establecerán canales de feedback para iterar y mejorar el reporte según las necesidades de los usuarios.

### 2.5. Implementación del Reporte

A continuación, implementaremos el reporte utilizando Python y las bibliotecas de visualización. El objetivo es generar un conjunto de gráficos y resúmenes que respondan a los propósitos definidos, para luego exportarlos a un archivo HTML. Empezaremos con las visualizaciones clave.

### Visualizaciones y gráficos

Para que el usuario final (o usuarios) entiendan mejor el reporte de negocio, se presentan las siguientes visualizaciones para su mejor comprensión:

### Gráfico de Distribución de Precios por Tipo de Expedición (Boxplot)

Esta visualización nos permite ver la dispersión de precios y los valores atípicos para cada tipo de expedición. Es útil para identificar si hay rangos de precios coherentes o si algún tipo de expedición tiene una variabilidad de precios muy alta.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='expedition_type', y='price', hue='expedition_type', palette='viridis', legend=False)
plt.title('Distribución de Precios por Tipo de Expedición')
plt.xlabel('Tipo de Expedición')
plt.ylabel('Precio (libras esterlinas)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Disponibilidad Media por Tipo de Expedición

Este gráfico muestra la disponibilidad promedio (`availability_365`) para cada tipo de expedición. Es un indicador clave para entender qué tipos de alojamientos tienen mayor o menor potencial de ocupación a lo largo del año.

In [ ]:
avg_availability = df.groupby('expedition_type')['availability_365'].mean().reset_index()
plt.figure(figsize=(10, 6))
sns.barplot(data=avg_availability, x='expedition_type', y='availability_365', hue='expedition_type', palette='muted', legend=False)
plt.title('Disponibilidad Media (días/año) por Tipo de Expedición')
plt.xlabel('Tipo de Expedición')
plt.ylabel('Días Disponibles al Año')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Número Medio de Reseñas por Tipo de Expedición

Las reseñas son un proxy de la satisfacción y popularidad. Este gráfico nos da una idea de qué tipos de expedición generan más feedback de los usuarios, lo que puede correlacionarse con una mayor demanda o engagement.

In [ ]:
avg_reviews = df.groupby('expedition_type')['number_of_reviews'].mean().reset_index()
plt.figure(figsize=(10, 6))
sns.barplot(data=avg_reviews, x='expedition_type', y='number_of_reviews', hue='expedition_type', palette='pastel', legend=False)
plt.title('Número Medio de Reseñas por Tipo de Expedición')
plt.xlabel('Tipo de Expedición')
plt.ylabel('Media de Reseñas')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Top 10 Ubicaciones (Barrios) más Frecuentes

Este gráfico de barras muestra los 10 barrios con la mayor cantidad de alojamientos. Es útil para identificar las áreas con mayor concentración de la oferta y comprender dónde Atlas Horizon Group podría tener más opciones para sus expediciones o donde la competencia es más alta.

In [ ]:
top_10_neighbourhoods = df['neighbourhood'].value_counts().head(10).reset_index()
top_10_neighbourhoods.columns = ['neighbourhood', 'count']
plt.figure(figsize=(12, 7))
sns.barplot(data=top_10_neighbourhoods, x='neighbourhood', y='count', hue='neighbourhood', palette='deep', legend=False)
plt.title('Top 10 Ubicaciones (Barrios) más Frecuentes')
plt.xlabel('Barrio')
plt.ylabel('Número de Alojamientos')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### Mapa de Alojamientos por Tipo de Expedición y Precio

Esta visualización es crucial para entender la distribución geográfica de los alojamientos y cómo se relacionan con los tipos de expedición y los precios. Permite a Atlas Horizon Group identificar áreas clave, concentraciones de ofertas o posibles huecos geográficos.

In [ ]:
# Importar librerías necesarias para el siguiente gráfico
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_map = px.scatter_mapbox(df, lat="latitude", lon="longitude", color="expedition_type", size="price",
                            color_continuous_scale=px.colors.cyclical.IceFire, size_max=15, zoom=9,
                            mapbox_style="carto-positron",
                            hover_name="name", hover_data=['price', 'room_type', 'expedition_type', 'neighbourhood'],
                            title='Distribución Geográfica de Alojamientos por Tipo de Expedición y Precio')
fig_map.update_layout(margin={"r":0,"t":50,"l":0,"b":0})
fig_map.show()

## 3. Generación del Dataset Limpio y el Informe HTML

### 3.1. Guardar el Dataset Limpio

In [ ]:
# Guardar el DataFrame limpio en un nuevo archivo CSV
df.to_csv('listings_feb_2020_cleaned.csv', index=False)
print('Dataset limpio guardado como listings_feb_2020_cleaned.csv')

### 3.2. Generar el Informe HTML Completo



In [ ]:
!jupyter nbconvert --to html '/content/P2_BRAHIM_RAUL_BI.ipynb'